In [ ]:
import pandas as pd

def clean_data(df):

    # Drop column: 'Title'
    df = df.drop(columns=['Title'])

    # Replace missing values with 0 in column: 'referral_ID'
    df = df.fillna({'referral_ID': 0})

    # Change column type to int64 for column: 'referral_ID'
    df = df.astype({'referral_ID': 'int64'})

    # Replace missing values with 0 in column: 'port_referral_ID'
    df = df.fillna({'port_referral_ID': 0})

    # Change column type to int64 for column: 'port_referral_ID'
    df = df.astype({'port_referral_ID': 'int64'})

    # Replace missing values with 0 in column: 'patient_ID'
    df = df.fillna({'patient_ID': 0})

    # Change column type to int64 for column: 'patient_ID'
    df = df.astype({'patient_ID': 'int64'})

    # Fill missing values in encounter_time with '0000' and convert to datetime
    df['encounter_time'] = df['encounter_time'].fillna('0000')
    df['encounter_time'] = df['encounter_time'].astype(str).str.replace('.0', '')
    df['encounter_time'] = pd.to_datetime(df['encounter_time'], format='%H%M').dt.time

    # Change column type to string for column: 'encounter_time'
    df = df.astype({'encounter_time': 'string'})

    # Replace encounter_date time with encounter_time if time is 00:00:00
    df['encounter_date'] = df.apply(
        lambda row: row['encounter_date'].replace(hour=int(row['encounter_time'][:2]),
                                                  minute=int(row['encounter_time'][3:5]),
                                                  second=int(row['encounter_time'][6:8]))
        if row['encounter_date'].time() == pd.Timestamp('00:00:00').time() else row['encounter_date'],
        axis=1
    )

    # Drop columns: 'encounter_time', 'narrative' and 7 other columns
    df = df.drop(columns=['encounter_time', 'narrative', 'encounter_narrative', 'nick_name', 'first_name', 'last_name', 'birthdate', 'age', 'encounter_personnel'])

    # Drop column: 'pcp_agency_category'
    df = df.drop(columns=['pcp_agency_category'])

    # Split text using string ';#' in column: 'encounter_agency'
    loc_0 = df.columns.get_loc('encounter_agency')
    df_split = df['encounter_agency'].str.split(pat=';#', expand=True, n=1).add_prefix('encounter_agency_')
    df = pd.concat([df.iloc[:, :loc_0], df_split, df.iloc[:, loc_0:]], axis=1)
    df = df.drop(columns=['encounter_agency'])

    # Drop columns: 'encounter_agency_1', 'encounter_business' and 7 other columns
    df = df.drop(columns=['encounter_agency_1', 'encounter_business', 'encounter_entity', 'Created', 'update_type_cat1', 'update_type_cat2', 'Modified', 'Item Type', 'Path'])

    # Fill missing pcp_agency with encounter_agency_0
    df['pcp_agency'] = df['pcp_agency'].fillna(df['encounter_agency_0'])

    # Replace missing values with "No data" in column: 'pcp_agency'
    df = df.fillna({'pcp_agency': "No data"})

    # Drop column: 'encounter_agency_0'
    df = df.drop(columns=['encounter_agency_0'])

    # Change column type to string for column: 'pcp_agency'
    df = df.astype({'pcp_agency': 'string'})

    df["pcp_agency"] = df["pcp_agency"].replace(
        {
            "Case Management N": "NOHN - Case Management",
            "CCHHS HRC": "CCHHS - Harm Reduction Center",
            "Medical N": "NOHN - Medical",
            "Medical Respite": "OPCC - Medical Respite",
            "OBOT N": "NOHN - OBOT",
            "Behavioral N": "NOHN - Behavioral",
            "Dental N": "NOHN - Dental",
            "O- Outreach": "OPCC - Outreach",
            "O - Other": "OPCC - Other",
            "O - MOUD": "OPCC - MOUD",
            "O- LEAD FIRE": "OPCC - LEAD FIRE",
            "O- Medical Respite": "OPCC - Medical Respite",
            "O3A": "Olympic Area Agency on Aging",
            "Other": "Other Organization",
            "Patient friend/acquaintance ": "Patient Family/caregiver",
            "REdisCOVERY": "OPCC - REdisCOVERY",
            "Reflections": "Reflections Counseling Services",
            "Station Walk-In": "911 Call/Walk-In",
        }
    )

    # Replace missing values with "No data" in column: 'pcp_agency'
    df = df.fillna({'pcp_agency': "No data"})

    # Replace missing values with "No data" in column: 'encounter_type_cat1'
    df = df.fillna({'encounter_type_cat1': "No data"})

    # Change column type to string for column: 'encounter_type_cat1'
    df = df.astype({'encounter_type_cat1': 'string'})

    # Replace missing values with "No data" in column: 'encounter_type_cat2'
    df = df.fillna({'encounter_type_cat2': "No data"})

    # Change column type to string for column: 'encounter_type_cat2'
    df = df.astype({'encounter_type_cat2': 'string'})

    # Replace missing values with "No data" in column: 'encounter_type_cat3'
    df = df.fillna({'encounter_type_cat3': "No data"})

    # Change column type to string for column: 'encounter_type_cat3'
    df = df.astype({'encounter_type_cat3': 'string'})

    # Replace missing values with "No data" in column: 'encounter_stage'
    df = df.fillna({'encounter_stage': "No data"})

    # Change column type to string for column: 'encounter_stage'
    df = df.astype({'encounter_stage': 'string'})

    return df

# Loaded variable 'df' from URI: c:\Users\imjus\dev\cpmdash\static\data\encounters.xlsx
df = pd.read_excel('encounters.xlsx')

df_clean = clean_data(df.copy())

# Save the cleaned data to a new csv file
df_clean.to_csv('encounters_clean.csv', index=False)